## 1. Setup

In [1]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)

# Take a look at the first few rows
print(df.head())


  Customer          ST GENDER             Education Customer Lifetime Value  \
0  RB50392  Washington    NaN                Master                     NaN   
1  QZ44356     Arizona      F              Bachelor              697953.59%   
2  AI49188      Nevada      F              Bachelor             1288743.17%   
3  WW63253  California      M              Bachelor              764586.18%   
4  GA49547  Washington      M  High School or Below              536307.65%   

    Income  Monthly Premium Auto Number of Open Complaints     Policy Type  \
0      0.0                1000.0                    1/0/00   Personal Auto   
1      0.0                  94.0                    1/0/00   Personal Auto   
2  48767.0                 108.0                    1/0/00   Personal Auto   
3      0.0                 106.0                    1/0/00  Corporate Auto   
4  36357.0                  68.0                    1/0/00   Personal Auto   

   Vehicle Class  Total Claim Amount  
0  Four-Door Car 

## 2.Challenge 1: Understanding the data

In [2]:
# dimension (raw, columns)
print("Shape of dataset:", df.shape)


Shape of dataset: (4008, 11)


In [3]:
# uniquee values per column
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

Customer: 1071 unique values
ST: 8 unique values
GENDER: 5 unique values
Education: 6 unique values
Customer Lifetime Value: 1027 unique values
Income: 774 unique values
Monthly Premium Auto: 132 unique values
Number of Open Complaints: 6 unique values
Policy Type: 3 unique values
Vehicle Class: 6 unique values
Total Claim Amount: 761 unique values


In [4]:
# catagorical vs numerical discriptions
print("Summary stats for numerical columns:\n", df.describe())
print("Summary stats for categorical columns:\n", df.describe(include='object'))


Summary stats for numerical columns:
              Income  Monthly Premium Auto  Total Claim Amount
count   1071.000000           1071.000000         1071.000000
mean   39295.701214            193.234360          404.986909
std    30469.427060           1601.190369          293.027260
min        0.000000             61.000000            0.382107
25%    14072.000000             68.000000          202.157702
50%    36234.000000             83.000000          354.729129
75%    64631.000000            109.500000          532.800000
max    99960.000000          35354.000000         2893.239678
Summary stats for categorical columns:
        Customer      ST GENDER Education Customer Lifetime Value  \
count      1071    1071    954      1071                    1068   
unique     1071       8      5         6                    1027   
top     RB50392  Oregon      F  Bachelor              445811.34%   
freq          1     320    457       324                       4   

       Number of Open C

In [7]:
# Check ranges for numerical columns ( income, customer lifetime value,total amount claim)
# Clean 'Customer Lifetime Value'
df["Customer Lifetime Value"] = (
    df["Customer Lifetime Value"]
    .astype(str)                           # make sure everything is string
    .str.replace("[\$,]", "", regex=True)  # remove $ and commas
)

# Convert to numeric (force errors to NaN)
df["Customer Lifetime Value"] = pd.to_numeric(df["Customer Lifetime Value"], errors="coerce")

# Now check ranges
print("Income range:", df["Income"].min(), "-", df["Income"].max())
print("CLV range:", df["Customer Lifetime Value"].min(), "-", df["Customer Lifetime Value"].max())
print("Total Claim Amount range:", df["Total Claim Amount"].min(), "-", df["Total Claim Amount"].max())



Income range: 0.0 - 99960.0
CLV range: nan - nan
Total Claim Amount range: 0.382107 - 2893.239678


## challenge 2- analysing the data

In [9]:
print(df.columns.tolist())


['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']


In [10]:
# Top 5 less common locations
# Top 5 less common locations
top5_locations = df["ST"].value_counts(ascending=True).head(5) # ST represent State
print("Top 5 less common customer locations:\n", top5_locations)



Top 5 less common customer locations:
 ST
AZ             25
WA             30
Washington     81
Nevada         98
Cali          120
Name: count, dtype: int64


In [11]:
#Most common policy type
policy_counts = df["Policy Type"].value_counts()
print("Policy counts:\n", policy_counts)
print("Policy with highest number sold:", policy_counts.idxmax())


Policy counts:
 Policy Type
Personal Auto     780
Corporate Auto    234
Special Auto       57
Name: count, dtype: int64
Policy with highest number sold: Personal Auto


In [12]:
#Average income comparison
personal_auto = df.loc[df["Policy Type"] == "Personal Auto"]
corporate_auto = df.loc[df["Policy Type"] == "Corporate Auto"]

print("Average income Personal Auto:", personal_auto["Income"].mean())
print("Average income Corporate Auto:", corporate_auto["Income"].mean())


Average income Personal Auto: 38180.69871794872
Average income Corporate Auto: 41390.31196581197


In [13]:
# high claim customers
# 75th percentile threshold
threshold = df["Total Claim Amount"].quantile(0.75)

# Filter customers above threshold
high_claims = df[df["Total Claim Amount"] > threshold]

print("High claim customers (top 25%):\n", high_claims)
print("Summary stats for high claim customers:\n", high_claims.describe())


High claim customers (top 25%):
      Customer          ST GENDER Education  Customer Lifetime Value   Income  \
1     QZ44356     Arizona      F  Bachelor                      NaN      0.0   
2     AI49188      Nevada      F  Bachelor                      NaN  48767.0   
17    OE15005        Cali    NaN   College                      NaN  28855.0   
23    TZ98966      Nevada    NaN  Bachelor                      NaN      0.0   
26    US89481  California    NaN  Bachelor                      NaN      0.0   
...       ...         ...    ...       ...                      ...      ...   
1059  YG44474      Oregon      M   College                      NaN  54193.0   
1061  RY92647        Cali      F  Bachelor                      NaN      0.0   
1068  GS98873     Arizona      F  Bachelor                      NaN  16061.0   
1069  CW49887  California      F    Master                      NaN  79487.0   
1070  MY31220  California      F   College                      NaN  54230.0   

      